In [ ]:
from importlib.resources import as_file, files
import os
from pathlib import Path

from openghg_inversions.rhime import run

tutorial_output_path = Path(os.environ.get("OPENGHG_TUTORIAL_OUTPUT_PATH", "outputs"))
resource = files("openghg_inversions.rhime").joinpath("config/multisector_tutorial.ini")
with as_file(resource) as config:
    result = run(model="multisector", config_file=config, output_path=tutorial_output_path)

{
    "OpenGHG Inversions commit": os.environ.get(
        "OPENGHG_TUTORIAL_CODE_REF", "local checkout"
    ),
    "tutorial data": os.environ.get("OPENGHG_TUTORIAL_DATA_TAG", "v1.0.0"),
    "sites": result.run_metadata["sites"],
    "observations": result.inv_inputs.sizes["nmeasure"],
    "posterior samples": {
        name: result.trace_group("posterior").sizes[name] for name in ("chain", "draw")
    },
}

{'OpenGHG Inversions commit': '9202222950c3ef2eb7bc83d040df3bcf6c26f94c',
 'tutorial data': 'v1.0.0',
 'sites': ['MHD', 'TAC'],
 'observations': 84,
 'posterior samples': {'chain': 2, 'draw': 50}}

In [ ]:
{
    "sectors": [
        (sector["name"], sector["flux_source"])
        for sector in result.model_metadata["sectors"]
    ],
    "H dimensions": result.inv_inputs["H"].dims,
    "source labels": result.inv_inputs["H"].source.values.tolist(),
    "flux state variable": result.variable_name("flux_scale"),
    "variable roles": result.model_metadata["variable_roles"],
}

{'sectors': [('anthropogenic', 'edgar-v80-anthropogenic'),
  ('wetlands', 'wetcharts-v131-wetlands')],
 'H dimensions': ('region', 'nmeasure', 'source'),
 'source labels': ['edgar-v80-anthropogenic', 'wetcharts-v131-wetlands'],
 'flux state variable': 'x',
 'variable roles': {'observation': 'mf',
  'observation_error': 'mf_error',
  'minimum_error': 'min_error',
  'concentration': 'y',
  'model_error': 'epsilon',
  'observation_repeatability': 'mf_repeatability',
  'observation_variability': 'mf_variability',
  'flux_scale': 'x',
  'flux_contribution': 'mu',
  'emissions_sensitivity': 'hx',
  'boundary': 'mu_bc',
  'baseline_scale': 'bc',
  'baseline_sensitivity': 'hbc',
  'baseline': 'mu_bc'}}

In [ ]:
from openghg_inversions.postprocessing.make_outputs import make_sector_flux_outputs

diagnostics = make_sector_flux_outputs(
    result,
    stats=["mean"],
    include_scale_factors=True,
    report_flux_on_inversion_grid=False,
)
product_stem = "multisector_tutorial" + result.start_date
diagnostic_path = tutorial_output_path / f"{product_stem}_sector_flux_diagnostics.nc"
inversion_path = tutorial_output_path / f"{product_stem}_inversion_output.nc"
assert diagnostic_path.exists() and inversion_path.exists()
{
    "diagnostic variables": sorted(diagnostics.data_vars),
    "diagnostic sizes": dict(diagnostics.sizes),
    "variable dimensions": {
        name: diagnostics[name].dims for name in sorted(diagnostics.data_vars)
    },
    "sector diagnostics file": diagnostic_path.name,
    "inversion output file": inversion_path.name,
}

{'diagnostic variables': ['flux_anthropogenic_posterior_mean',
  'flux_anthropogenic_prior_mean',
  'flux_total_posterior_mean',
  'flux_total_prior_mean',
  'flux_wetlands_posterior_mean',
  'flux_wetlands_prior_mean',
  'scaling_anthropogenic_posterior_mean',
  'scaling_anthropogenic_prior_mean',
  'scaling_wetlands_posterior_mean',
  'scaling_wetlands_prior_mean'],
 'diagnostic sizes': {'flux_time': 1, 'lat': 293, 'lon': 391},
 'variable dimensions': {'flux_anthropogenic_posterior_mean': ('flux_time',
   'lat',
   'lon'),
  'flux_anthropogenic_prior_mean': ('flux_time', 'lat', 'lon'),
  'flux_total_posterior_mean': ('flux_time', 'lat', 'lon'),
  'flux_total_prior_mean': ('flux_time', 'lat', 'lon'),
  'flux_wetlands_posterior_mean': ('flux_time', 'lat', 'lon'),
  'flux_wetlands_prior_mean': ('flux_time', 'lat', 'lon'),
  'scaling_anthropogenic_posterior_mean': ('lat', 'lon'),
  'scaling_anthropogenic_prior_mean': ('lat', 'lon'),
  'scaling_wetlands_posterior_mean': ('lat', 'lon'),
  